In [1]:
# 구글 웹검색 결과를 제공하는 serper dev

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from crewai import Agent, Task, Crew
from crewai.process import Process
# serp의 apikey를 가져와야 한다.
from crewai_tools import (
    SerperDevTool, # 특정 키워드를 줬을때, 해당 keyword를 구글에 검색한 결과를 가져오는 도구
    WebsiteSearchTool, # 특정 키워드, 웹사이트를 주면, 해당 웹사이트 내에서 해당 키워드와 관련이 있는 부분만 가져오는 RAG가 내부적으로 실행이 되는 도구
    ScrapeWebsiteTool # 해당 URL에 있는 모든 텍스트를 scrape해서 가져오는 도구. 일종의 crawling을 하는 도구다.
    
)

c:\Users\skyop\JaehoNote\.venv\Lib\site-packages\pydantic\_internal\_config.py:291: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.9/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
c:\Users\skyop\JaehoNote\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:547: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
c:\Users\skyop\JaehoNote\.venv\Lib\site-packages\crewai_tools\tools\scrapegraph_scrape_tool\scrapegraph_scrape_tool.py:34: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to 

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()
scrap_tool = ScrapeWebsiteTool()

# agent
researcher = Agent(
    role="테크 트렌드 연구원",
    goal="인공지능 분야의 최신 기술 트랜드를 한국어로 제공합니다. 지금은 2024년 9월입니다.",
    backstory="기술 트렌드에 예리한 한목을 지닌 분석가이자 AI개발자입니다.",
    tools=[search_tool, web_rag_tool, scrap_tool],
    verbose=True, # AI출력물들의 중간 결과값들
    max_iter=5, # 해당 agent가 주어진 task를 수행하는데 있어서 반복실행한 결과를 제한한다. 제한을 주지 않으면 계속해서 search하여 token수가 많이 나오게 된다.
    llm = llm
)

writer = Agent(
    role="뉴스레터 작성자",
    goal="최신 AI기술 트렌드에 대한 매력적인 테크 뉴스레터를 한국어로 작성하세요. 지금은 2024년 9월입니다.",
    backstory="기술에 대한 열정을 가진 숙련된 작가입니다.",
    verbose=True,
    allow_delegation=False, # True인경우, 해당 agent가 주어진 task를 수행하는데 있어서, 어려움을 겪는다면 다른 agent에게 해당 task를 위임하게된다.
    llm=llm
)

In [9]:
# task
research = Task(
    description="AI업게의 최신 기술 동향을 조사하고 요약을 제공하세요.",
    expected_output=("AI업계에서 가장 주목받는 3대 기술 개발동향과 그 중요성에 대한 신선한 관점을 요약한 글."),
    agent=researcher
)

write = Task(
    description="""테크 트랜드 연구원의 요약을 바탕으로 AI산업에 대한 매력적인 테크 뉴스레터를 작성하세요.
                테크 뉴스레터이므로 전문적인 용어를 사용해도 괜찮습니다. 답변에 있어서는 출처를 남겨주세요.
    """,
    expected_output="최신 기술 관련 소식을 재밌는 말투로 소개하는 4문단짜리 마크다운 형식 뉴스레터",
    agent=writer,
    output_file="./new_post.md"
)

In [10]:
# crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    verbose=True,
    process=Process.sequential
)
# execute task
result = crew.kickoff()

 
[2025-03-13 15:21:08][🚀 CREW 'CREW' STARTED, 12E53CBA-7EE6-4293-B105-E1076EA2B24E]: 2025-03-13 15:21:08.404485
 
[2025-03-13 15:21:08][📋 TASK STARTED: AI업게의 최신 기술 동향을 조사하고 요약을 제공하세요.]: 2025-03-13 15:21:08.416001
 
[2025-03-13 15:21:08][🤖 AGENT '테크 트렌드 연구원' STARTED TASK]: 2025-03-13 15:21:08.418002
# Agent: 테크 트렌드 연구원
## Task: AI업게의 최신 기술 동향을 조사하고 요약을 제공하세요.
 
[2025-03-13 15:21:08][🤖 LLM CALL STARTED]: 2025-03-13 15:21:08.418002
 
[2025-03-13 15:21:09][✅ LLM CALL COMPLETED]: 2025-03-13 15:21:09.318000
 
[2025-03-13 15:21:09][🤖 TOOL USAGE STARTED: 'SEARCH THE INTERNET WITH SERPER']: 2025-03-13 15:21:09.318999
 
[2025-03-13 15:21:10][✅ TOOL USAGE FINISHED: 'SEARCH THE INTERNET WITH SERPER']: 2025-03-13 15:21:10.092000


# Agent: 테크 트렌드 연구원
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"2024 AI technology trends\"}"
## Tool Output: 
{'searchParameters': {'q': '2024 AI technology trends', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{

In [7]:
from IPython.display import display, Markdown
display(Markdown(result.raw))

# 📬 최신 AI 기술 뉴스레터 (2024년 9월 호)

안녕하세요, AI 기술의 최신 동향을 사랑하는 여러분! 오늘은 2024년의 인공지능(AI) 업계에서 주목해야 할 세 가지 주요 트렌드를 소개해 드리겠습니다. AI가 우리 생활에 어떻게 혁신을 가져오고 있는지, 함께 알아보시죠!

## 🌟 멀티모달 AI의 시대

최근 AI 기술의 혁신 중 하나는 **멀티모달 AI**입니다. 이 기술은 텍스트, 이미지, 음성 등 다양한 데이터를 동시에 처리할 수 있는 능력을 가지고 있어요. 예를 들어, 여러분이 냉장고의 재료를 카메라로 비추며 요리법을 요청할 수 있는 상상을 해보세요! IBM의 Watsonx Assistant와 같은 시스템은 이러한 멀티모달 AI의 상용화 사례로, 기업들이 이를 통해 더 자연스러운 사용자 경험을 제공할 수 있게 도와줍니다. 뿐만 아니라, 시각 장애인을 위한 "Be My Eyes"와 같은 앱에서도 이 기술이 활용되고 있어, 정말 많은 사람들에게 도움을 주고 있답니다.

## ⚖️ 규제와 윤리적 AI의 중요성

AI 기술이 발전함에 따라 **규제 및 윤리적 문제**도 필수적으로 다루어져야 할 사항입니다. 최근 딥페이크와 같은 악용 사례가 증가하면서, AI 기술에 대한 규제의 필요성이 더욱 커지고 있습니다. 2023년 말에는 유럽 연합(EU)이 AI 법안에 대한 잠정 합의에 도달하였고, 이는 이미지 스크래핑을 통한 얼굴 인식 생성을 제한하는 내용을 포함하고 있습니다. 이러한 규제는 기업들이 AI 시스템을 도입하는 데 신중하게 접근하도록 유도하고 있으며, AI 기술의 채택 속도에도 영향을 미칠 것으로 예상됩니다.

## 🛠️ 작은 언어 모델과 오픈 소스의 발전

마지막으로 주목할 점은 **작은 언어 모델 및 오픈 소스의 발전**입니다. 2023년에는 여러 오픈 소스 언어 모델이 출시되면서, 기업들이 대규모 언어 모델을 사용할 수 있는 기회가 확대되었습니다. 예를 들어, Meta의 LLaMa와 같은 모델들은 상업적으로도 활용 가능해졌고, 이는 기존의 독점 모델보다 더 많은 사람들에게 접근 가능하다는 장점을 가지고 있습니다. 이러한 오픈 소스 모델들은 적은 수의 매개변수로도 높은 성능을 발휘할 수 있어, AI의 민주화를 촉진하고, 더 많은 기업들이 AI 기술을 활용할 수 있는 기반을 마련하고 있습니다.

2024년 AI 기술은 멀티모달 AI, 윤리적 규제, 오픈 소스의 발전을 통해 더욱 혁신적이고 포용적인 방향으로 나아갈 것입니다. 앞으로 어떤 변화가 있을지 기대되네요! 여러분, 다음 뉴스레터에서 또 만나요!

*출처: IBM, 유럽 연합 AI 법안, Meta*